# RK1828 Qwen3-ForcedAligner Demo


[Qwen3-ForcedAligner](https://github.com/QwenLM/Qwen3-ForcedAligner): given
**audio + known text**, output a timestamp per character/word (TSV), Chinese
and English, measured RTF 0.041. Typical uses: subtitle generation, lyrics
alignment, speech-data labeling.

| Step | Content |
|---|---|
| 1 | Download models and sample audio (from huggingface.co) |
| 2 | Extract the prebuilt demo |
| 3 | Enter the text and run |
| 4 | View the timestamps |

## 0. Environment check

Make sure the RKNN3 runtime (`librknn3_api.so`) and the RK1828 device are ready.
If this fails, install the driver/runtime first (the `rknn3-rk182x` DEB package)
and check with `sudo rknn-smi info` that the RK1828 shows up.

In [ ]:
from rknn3 import get_lib, find_devices

_ = get_lib()  # loads /usr/lib/librknn3_api.so; failure means the runtime is not installed

devices = find_devices()
if not devices:
    raise RuntimeError("No RK1828 device found! Check: sudo rknn-smi info")
for dev_id, dev_type in devices:
    print(f"Found device: id={dev_id}  interface={dev_type}")

## 1. Download models

The models for this demo are hosted on Hugging Face (account `qhddz`) and are
downloaded directly from **huggingface.co**. Run the next cell to start.

> Files that already exist (and are non-empty) are **skipped automatically** —
> re-running never downloads them twice.
> If your network cannot reach huggingface.co, set up a proxy first, e.g.
> `export https_proxy=http://<your-proxy>:<port>` before starting Jupyter.

In [ ]:
# (HF repo, path in repo, local path)
FILES = [
    ("qwen3-forced-aligner-rknn3-rk1828", "audio.wav", "demos/forced_aligner/audio.wav"),
    ("qwen3-forced-aligner-rknn3-rk1828", "demo_x86_64.tar.gz", "demos/forced_aligner/demo_x86_64.tar.gz"),
    ("qwen3-forced-aligner-rknn3-rk1828", "mel_128_filters.txt", "demos/forced_aligner/mel_128_filters.txt"),
    ("qwen3-forced-aligner-rknn3-rk1828", "model/Qwen3-ForcedAligner-0.6B_decoder.embed.bin", "demos/forced_aligner/model/Qwen3-ForcedAligner-0.6B_decoder.embed.bin"),
    ("qwen3-forced-aligner-rknn3-rk1828", "model/Qwen3-ForcedAligner-0.6B_decoder.tokenizer.gguf", "demos/forced_aligner/model/Qwen3-ForcedAligner-0.6B_decoder.tokenizer.gguf"),
    ("qwen3-forced-aligner-rknn3-rk1828", "model/Qwen3-ForcedAligner-0.6B_decoder_config.txt", "demos/forced_aligner/model/Qwen3-ForcedAligner-0.6B_decoder_config.txt"),
    ("qwen3-forced-aligner-rknn3-rk1828", "model/Qwen3-ForcedAligner-0.6B_decoder_rules.bin", "demos/forced_aligner/model/Qwen3-ForcedAligner-0.6B_decoder_rules.bin"),
    ("qwen3-forced-aligner-rknn3-rk1828", "model/decoder.rknn", "demos/forced_aligner/model/decoder.rknn"),
    ("qwen3-forced-aligner-rknn3-rk1828", "model/decoder.weight", "demos/forced_aligner/model/decoder.weight"),
    ("qwen3-forced-aligner-rknn3-rk1828", "model/encoder.rknn", "demos/forced_aligner/model/encoder.rknn"),
    ("qwen3-forced-aligner-rknn3-rk1828", "model/encoder.weight", "demos/forced_aligner/model/encoder.weight"),
]

In [ ]:
from nb_download import ensure_files

ensure_files(FILES, "hf")

## 2. Extract the prebuilt demo

`demo_x86_64.tar.gz` contains the official x86_64 C++ demo and its runtime
libraries (`lib/`). Hugging Face does not host raw executables, so they are
shipped as a tarball; extract once (re-running skips automatically).

> **Why a C++ demo instead of pure Python?**
> Notebooks 01-05 (classification / detection / FastSAM / pose / segmentation)
> are a single model with one forward pass and simple post-processing, so they
> walk through everything in Python (`rknn3.py`). This demo's pipeline is far
> more complex — chained models, tokenizer, KV cache, sliding-window /
> streaming state, hundreds of lines of orchestration — reimplementing it in
> Python would be costly and risk diverging from the official behavior.
> So we run the official `rknn3-model-zoo` C++ demo directly; source code:
> [github.com/airockchip/rknn3-model-zoo](https://github.com/airockchip/rknn3-model-zoo)
> under `examples/`.

In [ ]:
from nb_download import extract_tarball

extract_tarball("demos/forced_aligner/demo_x86_64.tar.gz", "demos/forced_aligner")

In [ ]:
from nb_demo import run_demo, snapshot, new_files, show_images, show_new_images, show_audio

## 3. Enter the text to align

The repo ships a sample `audio.wav` (Chinese). You can also use your own
wav + matching transcript.

In [ ]:
import ipywidgets as widgets
from IPython.display import display

w_wav = widgets.Text(value="audio.wav", description="Audio:",
                     layout=widgets.Layout(width="60%"))
w_text = widgets.Textarea(value="欢迎大家来到么哒社区进行体验",
                          description="Text:",
                          layout=widgets.Layout(width="70%", height="50px"))
w_lang = widgets.Dropdown(options=["Chinese", "English"], description="Language:")
display(w_wav, w_text, w_lang)
print("Then keep running downwards ▼")

In [ ]:
DEMO = "demos/forced_aligner"
before = snapshot(DEMO)
run_demo(["./rknn_qwen3_forced_aligner_demo",
          "--model_path", "./model",
          "--audio_path", w_wav.value,
          "--text", w_text.value,
          "--language", w_lang.value,
          "--mel_filters_path", "./mel_128_filters.txt",
          "--output_path", "./result.tsv"], DEMO)

## 4. View the timestamps (TSV: start s / end s / token)

In [ ]:
import csv

rows = list(csv.reader(open("demos/forced_aligner/result.tsv"), delimiter="\t"))
if rows and rows[0][0].lower().startswith("start"):
    rows = rows[1:]  # drop the header start_ms / end_ms / text
rows = [(float(r[0]) / 1000, float(r[1]) / 1000, r[2]) for r in rows]  # ms -> s
print(f"{'start(s)':>8} {'end(s)':>8}  token")
for s, e, t in rows:
    print(f"{s:8.2f} {e:8.2f}  {t}")

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import soundfile as sf
from vision_post import setup_cjk_font

setup_cjk_font()

wav_path = "demos/forced_aligner/audio.wav"
data, sr = sf.read(wav_path)
t = np.arange(len(data)) / sr
plt.figure(figsize=(12, 3))
plt.plot(t, data, lw=0.3, color="#888")
for s, e, txt in rows:
    plt.axvspan(s, e, alpha=0.25)
    plt.text((s + e) / 2, 0.8, txt, ha="center", fontsize=9)
plt.title("alignment over the waveform")
plt.xlabel("seconds")
plt.tight_layout()
plt.show()

## Summary

- Alignment is not recognition: the text is known, the model only **pins each
  token onto the timeline**.
- The waveform view makes the span-to-sound correspondence clearly visible.